In [6]:
#!/usr/bin/env python3
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import matplotlib.patches as mpatches

# — Edit these paths as needed —
datastruct_pkl = "/home/lq53/mir_repos/dappy_24_nov/2505_rerun/datastruct.p"  # your .p file
label_csv      = "/home/lq53/mir_repos/dappy_24_nov/2505_rerun/mir_label_rip.csv"
output_file    = "groom_clusters.png"                                        # where to save the figure

# 1. Load the watershed and borders
with open(datastruct_pkl, "rb") as f:
    data_obj = pickle.load(f)
ws_map  = data_obj.ws.watershed_map.astype(int)
borders = data_obj.ws.borders  # array of [x, y] coordinates for all boundaries

# 2. Manually parse the CSV (split on first comma only)
clusters = []
with open(label_csv, "r") as f:
    next(f)  # skip header
    for line in f:
        line = line.strip()
        if not line:
            continue
        cid_str, desc = line.split(",", 1)
        cid_str = cid_str.strip()
        desc    = desc.rstrip(",").strip()
        if not cid_str.isdigit():
            continue
        clusters.append((int(cid_str), desc))

df = pd.DataFrame(clusters, columns=["Cluster ID", "Description"])

# 3. Filter only grooming labels
mask      = df["Description"].str.contains("groom", case=False, na=False)
df_groom  = df[mask]

# 4. Build a map of groom types; non-groom clusters remain zero (white)
unique_descs = df_groom["Description"].unique().tolist()
groom_map    = np.zeros_like(ws_map)
for idx, desc in enumerate(unique_descs, start=1):
    clust_ids = df_groom[df_groom["Description"] == desc]["Cluster ID"].tolist()
    groom_map[np.isin(ws_map, clust_ids)] = idx

# 5. Prepare colormap: white background + one distinct color per groom type
colors = ["white"] + [plt.cm.tab10(i) for i in range(len(unique_descs))]
cmap   = ListedColormap(colors)

# 6. Plot the map
fig, ax = plt.subplots(figsize=(10, 10))
ax.imshow(groom_map, cmap=cmap, interpolation="nearest")
ax.set_axis_off()

# 7. Overlay watershed borders
ax.plot(borders[:, 0], borders[:, 1], ".", markersize=0.5, color="black")

# 8. Annotate every cluster with its ID at its centroid
for cid in np.unique(ws_map):
    if cid == 0:
        continue
    ys, xs = np.where(ws_map == cid)
    if ys.size == 0:
        continue
    y_mean = ys.mean()
    x_mean = xs.mean()
    ax.text(x_mean, y_mean, str(cid),
            ha="center", va="center",
            fontsize=4, color="black")

# 9. Legend for grooming types only
handles = [
    mpatches.Patch(color=colors[i+1], label=desc)
    for i, desc in enumerate(unique_descs)
]
ax.legend(
    handles=handles,
    title="Grooming types",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    borderaxespad=0.5
)

# 10. Save the figure
fig.savefig(output_file, bbox_inches="tight", dpi=300)
plt.close(fig)
print(f"Saved grooming clusters map with borders and labels to '{output_file}'")


Saved grooming clusters map with borders and labels to 'groom_clusters.png'


In [8]:
#!/usr/bin/env python3
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import matplotlib.patches as mpatches

# — Edit these paths as needed —
datastruct_pkl = "/home/lq53/mir_repos/dappy_24_nov/2505_rerun/datastruct.p"
label_csv      = "/home/lq53/mir_repos/dappy_24_nov/2505_rerun/mir_label_rip.csv"
output_file    = "all_labeled_clusters.png"

# 1. Load the watershed and borders
with open(datastruct_pkl, "rb") as f:
    data_obj = pickle.load(f)
ws_map  = data_obj.ws.watershed_map.astype(int)
borders = data_obj.ws.borders  # array of [x, y] coordinates for all boundaries

# 2. Read your labels CSV
clusters = []
with open(label_csv, "r") as f:
    next(f)  # skip header
    for line in f:
        line = line.strip()
        if not line:
            continue
        cid_str, desc = line.split(",", 1)
        cid_str = cid_str.strip()
        desc    = desc.rstrip(",").strip()
        if not cid_str.isdigit():
            continue
        clusters.append((int(cid_str), desc))

df = pd.DataFrame(clusters, columns=["Cluster ID", "Description"])

# 3. Build a map of all labeled clusters; others remain zero (white)
unique_descs = df["Description"].unique().tolist()
label_map    = np.zeros_like(ws_map)
for idx, desc in enumerate(unique_descs, start=1):
    mask_ids = df[df["Description"] == desc]["Cluster ID"].tolist()
    label_map[np.isin(ws_map, mask_ids)] = idx

# 4. Prepare colormap: white background + one color per label
colors = ["white"] + [plt.cm.tab20(i) for i in range(len(unique_descs))]
cmap   = ListedColormap(colors)

# 5. Plot the map
fig, ax = plt.subplots(figsize=(10, 10))
ax.imshow(label_map, cmap=cmap, interpolation="nearest")
ax.set_axis_off()

# 6. Overlay watershed borders
ax.plot(borders[:, 0], borders[:, 1], ".", markersize=0.5, color="black")

# 7. (Optional) annotate each cluster with its ID at its centroid
for cid in np.unique(ws_map):
    if cid == 0:
        continue
    ys, xs = np.where(ws_map == cid)
    if ys.size == 0:
        continue
    y_mean = ys.mean()
    x_mean = xs.mean()
    ax.text(x_mean, y_mean, str(cid),
            ha="center", va="center",
            fontsize=4, color="black")

# 8. Legend for all labels
handles = [
    mpatches.Patch(color=colors[i+1], label=desc)
    for i, desc in enumerate(unique_descs)
]
ax.legend(
    handles=handles,
    title="Labels",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    borderaxespad=0.5
)

# 9. Save the figure
fig.savefig(output_file, bbox_inches="tight", dpi=300)
plt.close(fig)
print(f"Saved labeled clusters map to '{output_file}'")


Saved labeled clusters map to 'all_labeled_clusters.png'


In [2]:
#!/usr/bin/env python3
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, to_rgba
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe

# --- paths ---
datastruct_pkl = "/home/lq53/mir_repos/dappy_24_nov/2505_rerun/datastruct.p"
label_csv      = "/home/lq53/mir_repos/dappy_24_nov/2505_rerun/mir_label_rip.csv"
output_file    = "all_labeled_clusters.png"   # also writes .svg

# --- output style (keep simple) ---
DPI            = 600
FIGSIZE        = (12, 12)
TRANSPARENT_BG = True
BORDER_ALPHA   = 0.5
ANNOTATE_IDS   = False   # True if you want per-cluster ID text
MAX_ANNOTATE   = 150
LEGEND_MAX_COLS= 3

# --- curated colors (edit here if you ever want different ones) ---
CURATED_COLORS = [
    "#4E79A7","#F28E2B","#E15759","#76B7B2","#59A14F",
    "#EDC948","#B07AA1","#FF9DA7","#9C755F","#BAB0AC",
    "#1F77B4","#FF7F0E","#2CA02C","#D62728","#9467BD",
    "#8C564B","#E377C2","#7F7F7F","#BCBD22","#17BECF"
]

# 1) load watershed + borders
with open(datastruct_pkl, "rb") as f:
    data_obj = pickle.load(f)
ws_map  = data_obj.ws.watershed_map.astype(int)
borders = np.asarray(data_obj.ws.borders)  # Nx2 (x,y)

# 2) read label csv -> (Cluster ID, Description)
rows = []
with open(label_csv, "r") as f:
    next(f, None)  # skip header
    for line in f:
        line = line.strip()
        if not line:
            continue
        cid_str, desc = line.split(",", 1)
        cid_str = cid_str.strip()
        desc    = desc.rstrip(",").strip()
        if cid_str.isdigit():
            rows.append((int(cid_str), desc))
df = pd.DataFrame(rows, columns=["Cluster ID", "Description"])

# 3) unique descriptions (preserve first appearance order)
unique_descs = df["Description"].drop_duplicates().tolist()

# map each description -> 1..K (0 = background)
label_map = np.zeros_like(ws_map, dtype=int)
for k, desc in enumerate(unique_descs, start=1):
    ids = df.loc[df["Description"] == desc, "Cluster ID"].to_numpy()
    if ids.size:
        label_map[np.isin(ws_map, ids)] = k

# 4) build colormap
k = len(unique_descs)
palette = (CURATED_COLORS * ((k + len(CURATED_COLORS) - 1) // len(CURATED_COLORS)))[:k]
palette = [to_rgba(c) for c in palette]

colors = [(0, 0, 0, 0)] + palette  # transparent background for label==0
cmap   = ListedColormap(colors, name="labels_cmap")

# 5) plot
fig, ax = plt.subplots(figsize=FIGSIZE)
ax.set_aspect("equal")
ax.set_axis_off()

masked = np.ma.masked_where(label_map == 0, label_map)
ax.imshow(masked, cmap=cmap, interpolation="none")  # crisp pixels

# 6) borders (sub-pixel points for crisp outlines)
if borders.size:
    ax.plot(borders[:, 0], borders[:, 1], ",", color="black", alpha=BORDER_ALPHA, lw=0, zorder=3)

# 7) optional ID annotations
if ANNOTATE_IDS:
    cids = np.unique(ws_map)
    cids = cids[cids != 0]
    if cids.size > MAX_ANNOTATE:
        rng = np.random.default_rng(0)
        cids = rng.choice(cids, size=MAX_ANNOTATE, replace=False)
    for cid in cids:
        ys, xs = np.where(ws_map == cid)
        if ys.size:
            x, y = xs.mean(), ys.mean()
            t = ax.text(x, y, str(cid), ha="center", va="center", fontsize=5, color="black", zorder=4)
            t.set_path_effects([pe.withStroke(linewidth=1.2, foreground="white")])

# 8) legend (compact columns)
if unique_descs:
    handles = [mpatches.Patch(color=palette[i], label=desc) for i, desc in enumerate(unique_descs)]
    n = len(unique_descs)
    ncol = min(LEGEND_MAX_COLS, max(1, int(np.ceil(n / 12))))
    leg = ax.legend(handles=handles, title="Labels", ncol=ncol, frameon=False,
                    bbox_to_anchor=(1.02, 1), loc="upper left", borderaxespad=0.5)
    plt.setp(leg.get_texts(), fontsize=8)
    plt.setp(leg.get_title(), fontsize=9, fontweight="bold")

# 9) save (PNG + SVG)
base, ext = os.path.splitext(output_file)
png_path = output_file if ext.lower() == ".png" else base + ".png"
svg_path = base + ".svg"
fig.savefig(png_path, dpi=DPI, bbox_inches="tight", transparent=TRANSPARENT_BG)
fig.savefig(svg_path, dpi=DPI, bbox_inches="tight", transparent=TRANSPARENT_BG)
plt.close(fig)
print(f"Saved: {png_path}")
print(f"Saved: {svg_path}")


Saved: all_labeled_clusters.png
Saved: all_labeled_clusters.svg


In [3]:
#!/usr/bin/env python3
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import matplotlib.patheffects as pe
import matplotlib.patches as mpatches
import os

# — Edit these paths as needed —
datastruct_pkl = "/home/lq53/mir_repos/dappy_24_nov/2505_rerun/datastruct.p"
label_csv      = "/home/lq53/mir_repos/dappy_24_nov/2505_rerun/mir_label_rip.csv"
output_file    = "all_labeled_clusters.png"   # will also write an .svg

# — Styling toggles —
ANNOTATE_IDS      = False          # poster usually cleaner without per-region IDs
MAX_ANNOTATE      = 150            # safety cap if you turn annotation on
BORDER_ALPHA      = 0.5            # 0..1; lower = subtler borders
BORDER_LW         = 0.3
DPI               = 600            # high DPI for print
FIGSIZE           = (12, 12)       # inches
LEGEND_MAX_COLS   = 3              # legend columns cap
TRANSPARENT_BG    = True           # transparent figure background for posters

# --------------------------------------------------------------------
# 1. Load the watershed and borders
with open(datastruct_pkl, "rb") as f:
    data_obj = pickle.load(f)
ws_map  = data_obj.ws.watershed_map.astype(int)
borders = np.asarray(data_obj.ws.borders)  # Nx2 [x,y]

# 2. Read labels CSV (Cluster ID, Description)
clusters = []
with open(label_csv, "r") as f:
    header = next(f, "")
    for line in f:
        line = line.strip()
        if not line:
            continue
        cid_str, desc = line.split(",", 1)
        cid_str = cid_str.strip()
        desc    = desc.rstrip(",").strip()
        if cid_str.isdigit():
            clusters.append((int(cid_str), desc))
df = pd.DataFrame(clusters, columns=["Cluster ID", "Description"])

# 3. Map each unique Description to an integer label (1..K); 0 stays background
unique_descs = df["Description"].tolist()
# keep first occurrence order, but ensure uniqueness
seen = set()
unique_descs = [d for d in unique_descs if not (d in seen or seen.add(d))]

label_map = np.zeros_like(ws_map, dtype=int)
for idx, desc in enumerate(unique_descs, start=1):
    ids = df.loc[df["Description"] == desc, "Cluster ID"].values
    if ids.size:
        label_map[np.isin(ws_map, ids)] = idx

# 4. Build a distinct color palette (background=transparent)
def make_palette(n_labels: int):
    # Concatenate several categorical maps for distinct colors (no new deps)
    cmaps = ["tab20", "tab20b", "tab20c", "Set3", "Paired"]
    pool = []
    for name in cmaps:
        cmap = plt.get_cmap(name)
        # Many categorical cmaps expose .colors; fallback to linspace sample
        if hasattr(cmap, "colors"):
            pool.extend(list(cmap.colors))
        else:
            pool.extend([cmap(x) for x in np.linspace(0, 1, cmap.N, endpoint=False)])
    if n_labels <= len(pool):
        return pool[:n_labels]
    # If more needed, fall back to evenly spaced HSV
    hsv = np.stack([np.linspace(0, 1, n_labels, endpoint=False),
                    np.ones(n_labels)*0.65,
                    np.ones(n_labels)*0.95], axis=1)
    from matplotlib.colors import hsv_to_rgb
    return hsv_to_rgb(hsv).tolist()

palette = make_palette(len(unique_descs))
# First entry is fully transparent for background (label 0)
colors = [(0, 0, 0, 0)] + [tuple(c) for c in palette]
cmap = ListedColormap(colors, name="labels_cmap")

# 5. Plot
fig, ax = plt.subplots(figsize=FIGSIZE)
ax.set_aspect("equal")
ax.set_axis_off()

# Mask background so figure background shows through
masked = np.ma.masked_where(label_map == 0, label_map)
im = ax.imshow(masked, cmap=cmap, interpolation="none")  # crisp edges

# 6. Overlay borders as fine, semi-transparent pixels
if borders.size > 0:
    # marker=',' draws sub-pixel points (very crisp)
    ax.plot(borders[:, 0], borders[:, 1], ",", color="black",
            alpha=BORDER_ALPHA, lw=0, zorder=3)

# 7. Optional: annotate cluster IDs at centroids with white stroke for legibility
if ANNOTATE_IDS:
    cids = np.unique(ws_map)
    cids = cids[cids != 0]
    if cids.size > MAX_ANNOTATE:
        cids = np.random.choice(cids, size=MAX_ANNOTATE, replace=False)
    # centroid per region (simple mean of pixel coordinates)
    for cid in cids:
        ys, xs = np.where(ws_map == cid)
        if ys.size == 0:
            continue
        x_mean, y_mean = xs.mean(), ys.mean()
        txt = ax.text(x_mean, y_mean, str(cid), ha="center", va="center",
                      fontsize=5, color="black", zorder=4)
        txt.set_path_effects([
            pe.withStroke(linewidth=1.2, foreground="white")
        ])

# 8. Legend (compact, multi-column)
if unique_descs:
    # Build handles aligned with palette order
    handles = [mpatches.Patch(color=palette[i], label=desc)
               for i, desc in enumerate(unique_descs)]
    # Columns: scale with count, but capped
    n = len(unique_descs)
    ncol = min(LEGEND_MAX_COLS, max(1, int(np.ceil(n / 12))))
    leg = ax.legend(handles=handles,
                    title="Labels",
                    ncol=ncol,
                    frameon=False,
                    bbox_to_anchor=(1.02, 1),
                    loc="upper left",
                    borderaxespad=0.5)
    # smaller legend font if many entries
    plt.setp(leg.get_texts(), fontsize=8)
    plt.setp(leg.get_title(), fontsize=9, fontweight="bold")

# 9. Save (PNG + SVG)
base, ext = os.path.splitext(output_file)
png_path = output_file if ext.lower() == ".png" else base + ".png"
svg_path = base + ".svg"

fig.savefig(png_path, dpi=DPI, bbox_inches="tight", transparent=TRANSPARENT_BG)
fig.savefig(svg_path, dpi=DPI, bbox_inches="tight", transparent=TRANSPARENT_BG)
plt.close(fig)
print(f"Saved: {png_path}")
print(f"Saved: {svg_path}")


Saved: all_labeled_clusters.png
Saved: all_labeled_clusters.svg


In [4]:
#!/usr/bin/env python3
import os, pickle, numpy as np, pandas as pd, matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, to_rgb
import matplotlib.patches as mpatches
import matplotlib.patheffects as pe
import colorsys

# --- paths ---
datastruct_pkl = "/home/lq53/mir_repos/dappy_24_nov/2505_rerun/datastruct.p"
label_csv      = "/home/lq53/mir_repos/dappy_24_nov/2505_rerun/mir_label_rip.csv"
output_file    = "all_labeled_clusters.png"   # also writes .svg

# --- output style ---
DPI, FIGSIZE = 600, (12, 12)
TRANSPARENT_BG = True
BORDER_ALPHA   = 0.45
ANNOTATE_IDS   = False
MAX_ANNOTATE   = 150
LEGEND_MAX_COLS= 3

# --- pastel controls (uniform look) ---
PASTEL_S = 0.42   # saturation (0–1)
PASTEL_L = 0.78   # lightness  (0–1)

# Base distinct hues (Tableau-ish). We’ll re-map to uniform S/L below.
BASE_HEX = [
    "#4E79A7","#F28E2B","#E15759","#76B7B2","#59A14F",
    "#EDC948","#B07AA1","#FF9DA7","#9C755F","#BAB0AC",
    "#1F77B4","#FF7F0E","#2CA02C","#D62728","#9467BD",
    "#8C564B","#E377C2","#7F7F7F","#BCBD22","#17BECF"
]

def pastelize(hex_list, target_s=PASTEL_S, target_l=PASTEL_L):
    """Keep each hue; set uniform saturation & lightness."""
    out = []
    for hx in hex_list:
        r, g, b = to_rgb(hx)
        h, l, s = colorsys.rgb_to_hls(r, g, b)   # NOTE: colorsys uses HLS
        r2, g2, b2 = colorsys.hls_to_rgb(h, target_l, target_s)
        out.append((r2, g2, b2, 1.0))
    return out

def build_pastel_palette(n_labels):
    base = pastelize(BASE_HEX)
    if n_labels <= len(base):
        return base[:n_labels]
    # Need more: generate evenly spaced hues at same S/L (keeps uniformity)
    hues = np.linspace(0, 1, n_labels, endpoint=False)
    extra = [(*colorsys.hls_to_rgb(h, PASTEL_L, PASTEL_S), 1.0) for h in hues]
    return extra

# 1) load data
with open(datastruct_pkl, "rb") as f:
    data_obj = pickle.load(f)
ws_map  = data_obj.ws.watershed_map.astype(int)
borders = np.asarray(data_obj.ws.borders)

# 2) labels csv
rows = []
with open(label_csv, "r") as f:
    next(f, None)
    for line in f:
        line = line.strip()
        if not line: continue
        cid_str, desc = line.split(",", 1)
        cid_str = cid_str.strip()
        desc    = desc.rstrip(",").strip()
        if cid_str.isdigit():
            rows.append((int(cid_str), desc))
df = pd.DataFrame(rows, columns=["Cluster ID", "Description"])

# 3) unique descriptions, first-appearance order
unique_descs = df["Description"].drop_duplicates().tolist()

# map description -> 1..K
label_map = np.zeros_like(ws_map, dtype=int)
for k, desc in enumerate(unique_descs, start=1):
    ids = df.loc[df["Description"] == desc, "Cluster ID"].to_numpy()
    if ids.size:
        label_map[np.isin(ws_map, ids)] = k

# 4) pastel colormap
k = len(unique_descs)
palette = build_pastel_palette(k)
colors  = [(0, 0, 0, 0)] + palette    # transparent background
cmap    = ListedColormap(colors, name="labels_pastel")

# 5) plot
fig, ax = plt.subplots(figsize=FIGSIZE)
ax.set_aspect("equal"); ax.set_axis_off()
masked = np.ma.masked_where(label_map == 0, label_map)
ax.imshow(masked, cmap=cmap, interpolation="none")

# 6) borders
if borders.size:
    ax.plot(borders[:, 0], borders[:, 1], ",", color="black",
            alpha=BORDER_ALPHA, lw=0, zorder=3)

# 7) optional IDs
if ANNOTATE_IDS:
    cids = np.unique(ws_map); cids = cids[cids != 0]
    if cids.size > MAX_ANNOTATE:
        rng = np.random.default_rng(0)
        cids = rng.choice(cids, size=MAX_ANNOTATE, replace=False)
    for cid in cids:
        ys, xs = np.where(ws_map == cid)
        if ys.size:
            x, y = xs.mean(), ys.mean()
            t = ax.text(x, y, str(cid), ha="center", va="center",
                        fontsize=5, color="black", zorder=4)
            t.set_path_effects([pe.withStroke(linewidth=1.2, foreground="white")])

# 8) legend
if unique_descs:
    handles = [mpatches.Patch(color=palette[i], label=desc)
               for i, desc in enumerate(unique_descs)]
    n = len(unique_descs)
    ncol = min(LEGEND_MAX_COLS, max(1, int(np.ceil(n / 12))))
    leg = ax.legend(handles=handles, title="Labels", ncol=ncol, frameon=False,
                    bbox_to_anchor=(1.02, 1), loc="upper left", borderaxespad=0.5)
    plt.setp(leg.get_texts(), fontsize=8)
    plt.setp(leg.get_title(), fontsize=9, fontweight="bold")

# 9) save
base, ext = os.path.splitext(output_file)
png_path = output_file if ext.lower() == ".png" else base + ".png"
svg_path = base + ".svg"
fig.savefig(png_path, dpi=DPI, bbox_inches="tight", transparent=TRANSPARENT_BG)
fig.savefig(svg_path, dpi=DPI, bbox_inches="tight", transparent=TRANSPARENT_BG)
plt.close(fig)
print(f"Saved: {png_path}")
print(f"Saved: {svg_path}")


Saved: all_labeled_clusters.png
Saved: all_labeled_clusters.svg
